# End-to-end Learning with Autoencoders — Rayleigh / Pilot-CSI / CNN

按依赖顺序重排：**所有定义在前（第 1-6 节），所有执行在后（第 7 节起）**。Run All 可一键跑通。

本周成果：块衰落 + 导频估计 CSI → 诊断并修复 U 形 BER → 达标 1e-2 → P 扫描找最优 → CNN 对比。

## 1. 导入与参数

In [ ]:
# Import Sionna
try:
    import sionna.phy
except ImportError as e:
    import os
    import sys
    if 'google.colab' in sys.modules:
       # Install Sionna in Google Colab
       print("Installing Sionna and restarting the runtime. Please run the cell again.")
       os.system("pip install sionna")
       os.kill(os.getpid(), 5)
    else:
       raise e

import pickle

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch._dynamo

from sionna.phy import Block
from sionna.phy.channel import AWGN
from sionna.phy.utils import ebnodb2no, expand_to_rank, sim_ber
from sionna.phy.fec.ldpc import LDPC5GEncoder, LDPC5GDecoder
from sionna.phy.mapping import Mapper, Demapper, Constellation, BinarySource

sionna.phy.config.seed = 42  # Set seed for reproducible random number generation

%matplotlib inline

In [ ]:
###############################################
# SNR range for evaluation and training [dB]
###############################################
ebno_db_min = 4.0
ebno_db_max = 8.0

###############################################
# Modulation and coding configuration
###############################################
num_bits_per_symbol = 6  # Baseline is 64-QAM
modulation_order = 2**num_bits_per_symbol
coderate = 0.5  # Coderate for the outer code
n = 1500  # Codeword length [bit]. Must be a multiple of num_bits_per_symbol
num_symbols_per_codeword = n // num_bits_per_symbol  # Number of modulated baseband symbols per codeword
k = int(n * coderate)  # Number of information bits per codeword

###############################################
# Training configuration
###############################################
num_training_iterations_conventional = 10000  # Number of training iterations for conventional training
# Number of training iterations with RL-based training for the alternating training phase and fine-tuning of the receiver phase
num_training_iterations_rl_alt = 7000
num_training_iterations_rl_finetuning = 3000
training_batch_size = 128  # Training batch size
rl_perturbation_var = 0.01  # Variance of the perturbation used for RL-based training of the transmitter
model_weights_path_conventional_training = "awgn_autoencoder_weights_conventional_training"  # Filename to save the autoencoder weights once conventional training is done
model_weights_path_rl_training = "awgn_autoencoder_weights_rl_training"  # Filename to save the autoencoder weights once RL-based training is done

###############################################
# Evaluation configuration
###############################################
results_filename = "awgn_autoencoder_results"  # Location to save the results

## 2. 网络与端到端系统类（MLP demapper）

In [ ]:
class NeuralDemapper(nn.Module):
    """Neural network-based demapper with three dense layers and ReLU activation.

    The input of the demapper consists of a received sample y ∈ ℂ and the noise
    variance. The output is a vector of LLRs for each bit carried by a symbol.
    """

    def __init__(self):
        super().__init__()

        self._dense_1 = nn.Linear(3, 128)
        self._dense_2 = nn.Linear(128, 128)
        self._dense_3 = nn.Linear(128, num_bits_per_symbol)  # Output LLRs for every bit

    def forward(self, y, no):
        # Using log10 scale helps with the performance
        no_db = torch.log10(no)

        # AWGN passes no as [batch size, 1] and needs broadcasting.
        # Rayleigh passes a per-symbol no_eff that is already [batch size, num_symbols_per_codeword].
        if no_db.shape[1] == 1:
            no_db = no_db.expand(-1, num_symbols_per_codeword)

        z = torch.stack([y.real, y.imag, no_db], dim=2)  # [batch size, num_symbols_per_codeword, 3]

        llr = F.relu(self._dense_1(z))
        llr = F.relu(self._dense_2(llr))
        llr = self._dense_3(llr)  # [batch size, num_symbols_per_codeword, num_bits_per_symbol]

        return llr

In [ ]:
class E2ESystemConventionalTraining(nn.Module):
    """End-to-end communication system for conventional training.
    
    This system transmits bits modulated using a trainable constellation over
    an AWGN channel. The receiver uses a neural network-based demapper.
    """

    def __init__(self, training):
        super().__init__()
        
        self._training = training
        
        ################
        ## Transmitter
        ################
        self._binary_source = BinarySource()
        # To reduce the computational complexity of training, the outer code is not used when training
        if not self._training:
            # num_bits_per_symbol is required for the interleaver
            self._encoder = LDPC5GEncoder(k, n, num_bits_per_symbol)
        
        # Trainable constellation
        # We initialize a custom constellation with QAM points
        qam_points = Constellation("qam", num_bits_per_symbol).points
        self.points_r = nn.Parameter(qam_points.real.clone())
        self.points_i = nn.Parameter(qam_points.imag.clone())

        self.constellation = Constellation("custom",
                                           num_bits_per_symbol,
                                           points=torch.complex(self.points_r, self.points_i),
                                           normalize=True,
                                           center=True)
       
        
        self._mapper = Mapper(constellation=self.constellation)
        
        ################
        ## Channel
        ################
        self._channel = AWGN()
        
        ################
        ## Receiver
        ################
        # We use the previously defined neural network for demapping
        self._demapper = NeuralDemapper()
        # To reduce the computational complexity of training, the outer code is not used when training
        if not self._training:
            self._decoder = LDPC5GDecoder(self._encoder, hard_out=True)

    def forward(self, batch_size, ebno_db):
        
        # Update constellation points from trainable parameters (creates fresh graph)
        self.constellation.points = torch.complex(self.points_r, self.points_i)

        # If `ebno_db` is a scalar, a tensor with shape [batch size] is created
        if ebno_db.dim() == 0:
            ebno_db = ebno_db.expand(batch_size)
        no = ebnodb2no(ebno_db, num_bits_per_symbol, coderate)
        no = expand_to_rank(no, 2)
        
        ################
        ## Transmitter
        ################
        # Outer coding is only performed if not training
        if self._training:
            c = self._binary_source([batch_size, n])
        else:
            b = self._binary_source([batch_size, k])
            c = self._encoder(b)
        # Modulation
        x = self._mapper(c)  # x [batch size, num_symbols_per_codeword]
        
        ################
        ## Channel
        ################
        y = self._channel(x, no)  # [batch size, num_symbols_per_codeword]
        
        ################
        ## Receiver
        ################
        llr = self._demapper(y, no)
        llr = llr.reshape(batch_size, n)
        # If training, outer decoding is not performed and the BCE is returned
        if self._training:
            loss = F.binary_cross_entropy_with_logits(llr, c)
            return loss
        else:
            # Outer decoding
            b_hat = self._decoder(llr)
            return b, b_hat  # Ground truth and reconstructed information bits returned for BER/BLER computation

In [ ]:
class E2ESystemRayleigh(E2ESystemConventionalTraining):
    """Same TX/RX, but over i.i.d. Rayleigh flat fading with perfect CSI."""

    def forward(self, batch_size, ebno_db):
        self.constellation.points = torch.complex(self.points_r, self.points_i)
        if ebno_db.dim() == 0:
            ebno_db = ebno_db.expand(batch_size)
        no = ebnodb2no(ebno_db, num_bits_per_symbol, coderate)
        no = expand_to_rank(no, 2)

        # Transmitter
        if self._training:
            c = self._binary_source([batch_size, n])
        else:
            b = self._binary_source([batch_size, k])
            c = self._encoder(b)
        x = self._mapper(c)

        # Channel: Rayleigh flat fading
        std = 0.5 ** 0.5
        h_r = torch.randn(x.shape, device=x.device, dtype=x.real.dtype) * std
        h_i = torch.randn(x.shape, device=x.device, dtype=x.real.dtype) * std
        h = torch.complex(h_r, h_i)
        y = self._channel(h * x, no)

        # Receiver: coherent equalization
        z = y / h
        no_eff = no / (h.real ** 2 + h.imag ** 2)

        llr = self._demapper(z, no_eff)
        llr = llr.reshape(batch_size, n)

        if self._training:
            loss = F.binary_cross_entropy_with_logits(llr, c)
            return loss
        else:
            b_hat = self._decoder(llr)
            return b, b_hat

## 3. 工具函数（训练 / 存 / 读 权重）

In [ ]:
def conventional_training(model):
    """Train the model using conventional SGD with backpropagation."""
    # Optimizer used to apply gradients
    optimizer = torch.optim.Adam(model.parameters())
    device = sionna.phy.config.device
    
    for i in range(num_training_iterations_conventional):
        optimizer.zero_grad()
        # Sampling a batch of SNRs
        ebno_db = torch.empty(training_batch_size, device=device).uniform_(ebno_db_min, ebno_db_max)
        # Forward pass
        loss = model(training_batch_size, ebno_db)
        # Computing and applying gradients
        loss.backward()
        optimizer.step()
        # Printing periodically the progress
        if i % 100 == 0:
            print(f'Iteration {i}/{num_training_iterations_conventional}  BCE: {loss.item():.4f}', end='\r')
    print()
    model.eval()
    optimizer.zero_grad(set_to_none=True)

In [ ]:
def save_weights(model, model_weights_path):
    m = getattr(model, "_orig_mod", model)
    torch.save(m.state_dict(), model_weights_path)

In [ ]:
# Utility function to load and set weights of a model
def load_weights(model, model_weights_path):
    """Load and set weights of a model."""
    device = sionna.phy.config.device
    state = torch.load(model_weights_path, map_location=device)
    model.load_state_dict(state, strict=False)
    # Update the constellation points from the loaded parameters
    model.constellation.points = torch.complex(model.points_r, model.points_i)

## 4. 基线类（AWGN + Rayleigh 完美 CSI）

In [ ]:
class Baseline(nn.Module):
    """Baseline system using QAM with Gray labeling and conventional demapping."""

    def __init__(self):
        super().__init__()
        
        ################
        ## Transmitter
        ################
        self._binary_source = BinarySource()
        self._encoder = LDPC5GEncoder(k, n, num_bits_per_symbol)
        constellation = Constellation("qam", num_bits_per_symbol)
        self.constellation = constellation
        self._mapper = Mapper(constellation=constellation)
        
        ################
        ## Channel
        ################
        self._channel = AWGN()
        
        ################
        ## Receiver
        ################
        self._demapper = Demapper("app", constellation=constellation)
        self._decoder = LDPC5GDecoder(self._encoder, hard_out=True)

    def forward(self, batch_size, ebno_db):
        # If `ebno_db` is a scalar, a tensor with shape [batch size] is created
        if ebno_db.dim() == 0:
            ebno_db = ebno_db.expand(batch_size)
        no = ebnodb2no(ebno_db, num_bits_per_symbol, coderate)
        no = expand_to_rank(no, 2)
        
        ################
        ## Transmitter
        ################
        b = self._binary_source([batch_size, k])
        c = self._encoder(b)
        # Modulation
        x = self._mapper(c)  # x [batch size, num_symbols_per_codeword]
        
        ################
        ## Channel
        ################
        y = self._channel(x, no)  # [batch size, num_symbols_per_codeword]
        
        ################
        ## Receiver
        ################
        llr = self._demapper(y, no)
        # Outer decoding
        b_hat = self._decoder(llr)
        return b, b_hat  # Ground truth and reconstructed information bits returned for BER/BLER computation

In [ ]:
class BaselineRayleigh(nn.Module):
    """Baseline over an i.i.d. Rayleigh flat-fading channel with perfect CSI."""

    def __init__(self):
        super().__init__()
        self._binary_source = BinarySource()
        self._encoder = LDPC5GEncoder(k, n, num_bits_per_symbol)
        constellation = Constellation("qam", num_bits_per_symbol)
        self.constellation = constellation
        self._mapper = Mapper(constellation=constellation)
        self._channel = AWGN()
        self._demapper = Demapper("app", constellation=constellation)
        self._decoder = LDPC5GDecoder(self._encoder, hard_out=True)

    def forward(self, batch_size, ebno_db):
        if ebno_db.dim() == 0:
            ebno_db = ebno_db.expand(batch_size)
        no = ebnodb2no(ebno_db, num_bits_per_symbol, coderate)
        no = expand_to_rank(no, 2)

        # Transmitter
        b = self._binary_source([batch_size, k])
        c = self._encoder(b)
        x = self._mapper(c)

        # Channel: Rayleigh flat fading, i.i.d. per symbol
        std = 0.5 ** 0.5
        h_r = torch.randn(x.shape, device=x.device, dtype=x.real.dtype) * std
        h_i = torch.randn(x.shape, device=x.device, dtype=x.real.dtype) * std
        h = torch.complex(h_r, h_i)
        y = self._channel(h * x, no)

        # Receiver: coherent equalization (perfect CSI)
        z = y / h
        no_eff = no / (h.real ** 2 + h.imag ** 2)

        llr = self._demapper(z, no_eff)
        b_hat = self._decoder(llr)
        return b, b_hat

## 5. 块衰落 + 导频估计（MLP）

In [ ]:
device = sionna.phy.config.device
print('device =', device)

In [ ]:
def make_rayleigh_block_frame(x_data, P, L, no, device, perfect_csi=False):
    """
    x_data: [B, num_data_symbols] 复数
    no:     标量或 [B,1] 噪声功率
    perfect_csi=True 时用真 h（上界对照）；False 时用导频估计 ĥ
    返回 z, no_eff, h, h_hat, num_frames, pad
    """
    B, num_data = x_data.shape
    data_per_frame = L - P
    num_frames = (num_data + data_per_frame - 1) // data_per_frame

    # --- 每帧一个独立 h ---
    std = 0.5 ** 0.5
    h = torch.complex(
        torch.randn(B, num_frames, 1, device=device) * std,
        torch.randn(B, num_frames, 1, device=device) * std,
    )  # [B, F, 1]

    # --- 数据符号分帧 ---
    pad = num_frames * data_per_frame - num_data
    x_padded = F.pad(x_data, (0, pad))
    x_framed = x_padded.reshape(B, num_frames, data_per_frame)  # [B, F, data_per_frame]

    if not torch.is_tensor(no):
        no = torch.tensor(float(no), device=device)
    no = no.reshape(-1, 1, 1) if no.ndim >= 1 else no.reshape(1, 1, 1)  # [B,1,1] 或 [1,1,1]，保证广播
    n_std = (no / 2.0).sqrt()

    def cn(shape):  # 复高斯噪声 CN(0, no)
        return torch.complex(torch.randn(shape, device=device) * n_std,
                             torch.randn(shape, device=device) * n_std)

    # --- 导频段：x_p = 1+0j，过同一个 h，估计 ĥ ---
    x_pilot = torch.ones(B, num_frames, P, device=device, dtype=x_data.dtype)  # 单位模长
    y_pilot = h * x_pilot + cn((B, num_frames, P))
    # LS 估计：ĥ = (1/P) Σ x̄_p y_p   （单位模长导频下就是平均）
    h_hat = (torch.conj(x_pilot) * y_pilot).mean(dim=2, keepdim=True)  # [B, F, 1]

    # --- 数据段：过真 h ---
    y_framed = h * x_framed + cn((B, num_frames, data_per_frame))

    # --- 均衡：用 ĥ（或真 h 做上界对照）---
    h_use = h if perfect_csi else h_hat
    z_framed = y_framed / h_use
    no_eff_framed = no / (h_use.real ** 2 + h_use.imag ** 2)
    no_eff_framed = no_eff_framed.expand(-1, -1, data_per_frame)

    # --- 拉平回符号流，去 padding ---
    z = z_framed.reshape(B, -1)[:, :num_data]
    no_eff = no_eff_framed.reshape(B, -1)[:, :num_data]

    return z, no_eff, h, h_hat, num_frames, pad

In [ ]:
class E2ESystemPilotCSI(E2ESystemConventionalTraining):
    """块衰落 + 导频估计 CSI。继承 TX/RX，复用 AWGN 权重。"""

    def __init__(self, P, L, training=False, perfect_csi=False):
        super().__init__(training=training)
        self.P = P
        self.L = L
        self.perfect_csi = perfect_csi

    def forward(self, batch_size, ebno_db):
        self.constellation.points = torch.complex(self.points_r, self.points_i)
        if ebno_db.dim() == 0:
            ebno_db = ebno_db.expand(batch_size)

        # 关键：导频开销折进有效码率
        coderate_eff = coderate * (self.L - self.P) / self.L
        no = ebnodb2no(ebno_db, num_bits_per_symbol, coderate_eff)
        no = expand_to_rank(no, 2)

        # 发射端
        if self._training:
            c = self._binary_source([batch_size, n])
        else:
            b = self._binary_source([batch_size, k])
            c = self._encoder(b)
        x = self._mapper(c)   # [B, 250]

        # 块衰落信道 + 导频估计（复用刚验证好的函数）
        no_scalar = no[:, :1]  # [B,1]
        z, no_eff, h, h_hat, nf, pad = make_rayleigh_block_frame(
            x, P=self.P, L=self.L, no=no_scalar,
            device=x.device, perfect_csi=self.perfect_csi)

        # 接收端
        llr = self._demapper(z, no_eff)
        llr = llr.reshape(batch_size, n)

        if self._training:
            return F.binary_cross_entropy_with_logits(llr, c)
        else:
            b_hat = self._decoder(llr)
            return b, b_hat

## 6. CNN demapper（帧感受野）

In [ ]:
def make_frame_cnn(x_data, P, L, no, device, perfect_csi=False):
    """返回帧结构张量供 CNN 用（不拉平）。
    features: [B, num_frames, data_per_frame, C]  C=5: Re(z),Im(z),Re(ĥ),Im(ĥ),log10(no_eff)
    """
    B, num_data = x_data.shape
    dpf = L - P
    nf = (num_data + dpf - 1) // dpf
    std = 0.5 ** 0.5
    h = torch.complex(torch.randn(B, nf, 1, device=device) * std,
                      torch.randn(B, nf, 1, device=device) * std)
    pad = nf * dpf - num_data
    x_framed = F.pad(x_data, (0, pad)).reshape(B, nf, dpf)

    if not torch.is_tensor(no):
        no = torch.tensor(float(no), device=device)
    no = no.reshape(-1, 1, 1) if no.ndim >= 1 else no.reshape(1, 1, 1)
    n_std = (no / 2.0).sqrt()
    def cn(shape):
        return torch.complex(torch.randn(shape, device=device) * n_std,
                             torch.randn(shape, device=device) * n_std)

    x_pilot = torch.ones(B, nf, P, device=device, dtype=x_data.dtype)
    y_pilot = h * x_pilot + cn((B, nf, P))
    h_hat = (torch.conj(x_pilot) * y_pilot).mean(dim=2, keepdim=True)  # [B,nf,1]

    y_framed = h * x_framed + cn((B, nf, dpf))
    h_use = h if perfect_csi else h_hat
    z = y_framed / h_use                                    # [B,nf,dpf]
    no_eff = (no / (h_use.real**2 + h_use.imag**2))         # [B,nf,1]

    # 组装通道，ĥ 和 no_eff 广播到每个符号
    h_use_b = h_use.expand(-1, -1, dpf)
    no_eff_b = no_eff.expand(-1, -1, dpf)
    feats = torch.stack([z.real, z.imag,
                         h_use_b.real, h_use_b.imag,
                         torch.log10(no_eff_b)], dim=-1)     # [B,nf,dpf,5]
    return feats, num_data, nf, pad, dpf

In [ ]:
class CNNDemapper(nn.Module):
    """帧内 1D 卷积，感受野覆盖相邻符号。"""
    def __init__(self, in_ch=5):
        super().__init__()
        self.c1 = nn.Conv1d(in_ch, 64, kernel_size=3, padding=1)
        self.c2 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.c3 = nn.Conv1d(64, num_bits_per_symbol, kernel_size=1)

    def forward(self, feats):
        # feats [B, nf, dpf, C] -> [B*nf, C, dpf]
        B, nf, dpf, C = feats.shape
        x = feats.reshape(B * nf, dpf, C).permute(0, 2, 1)
        x = F.relu(self.c1(x))
        x = F.relu(self.c2(x))
        x = self.c3(x)                       # [B*nf, 6, dpf]
        x = x.permute(0, 2, 1).reshape(B, nf, dpf, num_bits_per_symbol)
        return x

class E2ESystemPilotCNN(E2ESystemConventionalTraining):
    def __init__(self, P, L, training=False, perfect_csi=False):
        super().__init__(training=training)
        self.P = P; self.L = L; self.perfect_csi = perfect_csi
        self._demapper = CNNDemapper(in_ch=5)   # 覆盖掉 MLP demapper

    def forward(self, batch_size, ebno_db):
        self.constellation.points = torch.complex(self.points_r, self.points_i)
        if ebno_db.dim() == 0:
            ebno_db = ebno_db.expand(batch_size)
        coderate_eff = coderate * (self.L - self.P) / self.L
        no = ebnodb2no(ebno_db, num_bits_per_symbol, coderate_eff)
        no = expand_to_rank(no, 2)

        if self._training:
            c = self._binary_source([batch_size, n])
        else:
            b = self._binary_source([batch_size, k])
            c = self._encoder(b)
        x = self._mapper(c)

        feats, num_data, nf, pad, dpf = make_frame_cnn(
            x, self.P, self.L, no[:, :1], x.device, self.perfect_csi)
        llr_framed = self._demapper(feats)               # [B,nf,dpf,6]
        llr = llr_framed.reshape(batch_size, -1, num_bits_per_symbol)[:, :num_data, :]
        llr = llr.reshape(batch_size, n)

        if self._training:
            return F.binary_cross_entropy_with_logits(llr, c)
        else:
            b_hat = self._decoder(llr)
            return b, b_hat

## 7. 训练自编码器（AWGN，约 4-5 分钟）

得到 AWGN 权重，供后续所有 Rayleigh 实验热启动。

In [ ]:
# Instantiate and train the end-to-end system
model_weights_path_conventional_training = 'awgn_autoencoder_weights_conventional_training'
model = E2ESystemConventionalTraining(training=True).to(device)
conventional_training(model)
save_weights(model, model_weights_path_conventional_training)

## 8. 自检：LS 估计精度（误差 ≈ 理论 N0/P）

In [ ]:
# LS 估计精度自检：误差应 ≈ 理论 no/P
torch.manual_seed(0)
x_test = torch.randn(64, 250, dtype=torch.complex64, device=device)
no_test = 0.1
for P in [2, 4, 6, 8]:
    z, no_eff, h, h_hat, nf, pad = make_rayleigh_block_frame(
        x_test, P=P, L=25, no=no_test, device=device)
    err = (h - h_hat).abs().pow(2).mean().item()
    print(f'P={P}: MSE={err:.5f}   theory no/P={no_test/P:.5f}')

## 9. 自检：Pilot-CSI 前向能跑通

In [ ]:
m = E2ESystemPilotCSI(P=4, L=25, training=False, perfect_csi=False).to(device)
load_weights(m, model_weights_path_conventional_training)
ebno = torch.tensor(12.0, device=device)
b, b_hat = m(64, ebno)
print('b:', b.shape, 'b_hat:', b_hat.shape)
print('raw BER:', (b != b_hat).float().mean().item())

## 10. 诊断：U 形 BER 来自分布偏移

先看未重训的 Pilot-CSI 会出现 U 形（高 SNR 反而变差）；再用二分法（完美CSI vs BaselineRayleigh）定位到 demapper。

### 10a. 未重训扫描（会出现 U 形，约 15-20 分钟）

In [ ]:
import numpy as np

ebno_dbs_pilot = np.arange(8.0, 18.1, 1.0)   # 块衰落+导频，瀑布区更靠右
P_list = [2, 4, 6, 8]
BER_pilot = {}

for P in P_list:
    m = E2ESystemPilotCSI(P=P, L=25, training=False, perfect_csi=False).to(device)
    load_weights(m, model_weights_path_conventional_training)
    with torch.no_grad():
        ber, bler = sim_ber(m, ebno_dbs_pilot, batch_size=128,
                            num_target_block_errors=300, max_mc_iter=100)
    BER_pilot[f'P={P}'] = ber.cpu().numpy()
    print(f'--- P={P} done ---')

# 加一条完美 CSI 上界做对照
m = E2ESystemPilotCSI(P=4, L=25, training=False, perfect_csi=True).to(device)
load_weights(m, model_weights_path_conventional_training)
with torch.no_grad():
    ber, _ = sim_ber(m, ebno_dbs_pilot, batch_size=128,
                     num_target_block_errors=300, max_mc_iter=100)
BER_pilot['perfect CSI (P=4)'] = ber.cpu().numpy()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 6))
for label, ber in BER_pilot.items():
    ber = np.asarray(ber)
    mask = ber > 0
    ls = '--' if 'perfect' in label else '-'
    plt.semilogy(ebno_dbs_pilot[mask], ber[mask], marker='o', linestyle=ls, label=label)
plt.axhline(1e-2, color='gray', ls=':', label='target 1e-2')
plt.xlabel(r'$E_b/N_0$ (dB)')
plt.ylabel('BER')
plt.grid(which='both', alpha=0.4)
plt.legend()
plt.title('Block-fading + pilot CSI: BER vs pilot count P (L=25)')
plt.tight_layout()
plt.savefig('ber_vs_pilot.png', dpi=150)

### 10b. 二分法定位（完美CSI 神经网络 vs APP 基线）

In [ ]:
import numpy as np
for ebno in [10.0, 14.0, 18.0]:
    m = E2ESystemPilotCSI(P=4, L=25, training=False, perfect_csi=True).to(device)
    load_weights(m, model_weights_path_conventional_training)
    with torch.no_grad():
        ber, bler = sim_ber(m, np.array([ebno]), batch_size=256,
                            num_target_block_errors=500, max_mc_iter=200)
    print(f"ebno={ebno}: BER={ber.item():.4e}")

In [ ]:
for ebno in [10.0, 14.0, 18.0]:
    m = BaselineRayleigh().to(device)
    with torch.no_grad():
        ber, _ = sim_ber(m, np.array([ebno]), batch_size=256,
                        num_target_block_errors=500, max_mc_iter=200)
    print(f"Baseline ebno={ebno}: BER={ber.item():.4e}")

## 11. 修复：在 Rayleigh 上重训

隔离变量：先完美 CSI 重训（证明能修 demapper），再导频估计重训（最终形态）。

### 11a. 完美 CSI 重训 + 验证 U 形消失

In [ ]:
# 在块衰落 + 完美CSI 上重训（先不加导频，隔离变量）
model_rayleigh = E2ESystemPilotCSI(P=4, L=25, training=True, perfect_csi=True).to(device)
load_weights(model_rayleigh, model_weights_path_conventional_training)  # 从 AWGN 权重热启动

# 重训（复用 conventional_training，但 SNR 范围要对齐块衰落瀑布区）
import torch
optimizer = torch.optim.Adam(model_rayleigh.parameters(), lr=1e-3)
for it in range(3000):
    ebno_db = torch.empty(128, device=device).uniform_(8.0, 16.0)  # 对齐块衰落瀑布区
    optimizer.zero_grad()
    loss = model_rayleigh(128, ebno_db)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_rayleigh.parameters(), 1.0)  # 防深衰落梯度爆炸
    optimizer.step()
    if it % 500 == 0:
        print(f"it {it}: BCE={loss.item():.4f}")

save_weights(model_rayleigh, 'rayleigh_perfect_csi_weights')

In [ ]:
import numpy as np

print("=== 重训后（rayleigh_perfect_csi_weights）===")
for ebno in [10.0, 14.0, 18.0]:
    m = E2ESystemPilotCSI(P=4, L=25, training=False, perfect_csi=True).to(device)
    load_weights(m, 'rayleigh_perfect_csi_weights')       # ← 用新权重
    with torch.no_grad():
        ber, bler = sim_ber(m, np.array([ebno]), batch_size=256,
                            num_target_block_errors=500, max_mc_iter=200)
    print(f"ebno={ebno}: BER={ber.item():.4e}")

### 11b. 导频估计重训 + 验证达标（最终形态）

In [ ]:
# 最终形态：块衰落 + 导频估计 CSI，重训
model_pilot = E2ESystemPilotCSI(P=4, L=25, training=True, perfect_csi=False).to(device)
load_weights(model_pilot, model_weights_path_conventional_training)  # 从 AWGN 热启动

optimizer = torch.optim.Adam(model_pilot.parameters(), lr=1e-3)
for it in range(3000):
    ebno_db = torch.empty(128, device=device).uniform_(8.0, 16.0)
    optimizer.zero_grad()
    loss = model_pilot(128, ebno_db)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_pilot.parameters(), 1.0)
    optimizer.step()
    if it % 500 == 0:
        print(f"it {it}: BCE={loss.item():.4f}")

save_weights(model_pilot, 'rayleigh_pilot_csi_weights')

In [ ]:
import numpy as np

print("=== 导频估计 + 重训（rayleigh_pilot_csi_weights）===")
for ebno in [10.0, 14.0, 18.0]:
    m = E2ESystemPilotCSI(P=4, L=25, training=False, perfect_csi=False).to(device)
    load_weights(m, 'rayleigh_pilot_csi_weights')
    with torch.no_grad():
        ber, bler = sim_ber(m, np.array([ebno]), batch_size=256,
                            num_target_block_errors=500, max_mc_iter=200)
    print(f"ebno={ebno}: BER={ber.item():.4e}")

## 12. P 扫描：每个 P 各自重训，找最优导频数

约 20-25 分钟。结论：最优 P 取决于 SNR，交叉点约 11 dB。

In [ ]:
import numpy as np

ebno_dbs_final = np.arange(8.0, 16.1, 1.0)
P_list = [2, 4, 6, 8]
BER_final = {}
weights_by_P = {}

# 1) 每个 P 各自重训（从 AWGN 热启动）
for P in P_list:
    print(f"===== 重训 P={P} =====")
    mt = E2ESystemPilotCSI(P=P, L=25, training=True, perfect_csi=False).to(device)
    load_weights(mt, model_weights_path_conventional_training)
    opt = torch.optim.Adam(mt.parameters(), lr=1e-3)
    for it in range(2000):
        ebno_db = torch.empty(128, device=device).uniform_(8.0, 16.0)
        opt.zero_grad()
        loss = mt(128, ebno_db)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(mt.parameters(), 1.0)
        opt.step()
    wpath = f'rayleigh_pilot_weights_P{P}'
    save_weights(mt, wpath)
    weights_by_P[P] = wpath
    print(f"  P={P} 训练完 BCE={loss.item():.4f}")

# 2) 每个 P 评估 BER
for P in P_list:
    me = E2ESystemPilotCSI(P=P, L=25, training=False, perfect_csi=False).to(device)
    load_weights(me, weights_by_P[P])
    with torch.no_grad():
        ber, _ = sim_ber(me, ebno_dbs_final, batch_size=128,
                         num_target_block_errors=300, max_mc_iter=100)
    BER_final[f'P={P}'] = ber.cpu().numpy()
    print(f"--- P={P} 评估完 ---")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(9, 6))
for label, ber in BER_final.items():
    ber = np.asarray(ber)
    mask = ber > 0
    plt.semilogy(ebno_dbs_final[mask], ber[mask], marker='o', label=label)
plt.axhline(1e-2, color='gray', ls=':', label='target 1e-2')
plt.xlabel(r'$E_b/N_0$ (dB)')
plt.ylabel('BER')
plt.grid(which='both', alpha=0.4)
plt.legend()
plt.title('Retrained on Rayleigh + pilot CSI: BER vs pilot count P (L=25)')
plt.tight_layout()
plt.savefig('ber_vs_P_retrained.png', dpi=150)
plt.show()

## 13. CNN vs MLP 对比

结论：平坦块衰落下 CNN 与 MLP 基本持平（MLP 略优），因为均衡后符号间无空间相关性可挖。

In [ ]:
mc = E2ESystemPilotCNN(P=2, L=25, training=False, perfect_csi=False).to(device)
ebno = torch.tensor(12.0, device=device)
b, b_hat = mc(64, ebno)
print("b:", b.shape, "b_hat:", b_hat.shape)   # 期望 [64,750] [64,750]
print("能前向 ✓")

In [ ]:
# 训练 CNN demapper（P=2 最优，从零训）
model_cnn = E2ESystemPilotCNN(P=2, L=25, training=True, perfect_csi=False).to(device)

optimizer = torch.optim.Adam(model_cnn.parameters(), lr=1e-3)
for it in range(4000):                      # CNN 从零训，比 MLP 多给点步数
    ebno_db = torch.empty(128, device=device).uniform_(8.0, 16.0)
    optimizer.zero_grad()
    loss = model_cnn(128, ebno_db)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_cnn.parameters(), 1.0)
    optimizer.step()
    if it % 500 == 0:
        print(f"it {it}: BCE={loss.item():.4f}")

save_weights(model_cnn, 'rayleigh_cnn_P2_weights')

In [ ]:
import numpy as np
print("=== CNN P=2 ===")
for ebno in [10.0, 14.0, 18.0]:
    m = E2ESystemPilotCNN(P=2, L=25, training=False, perfect_csi=False).to(device)
    load_weights(m, 'rayleigh_cnn_P2_weights')
    with torch.no_grad():
        ber, _ = sim_ber(m, np.array([ebno]), batch_size=256,
                         num_target_block_errors=500, max_mc_iter=200)
    print(f"ebno={ebno}: BER={ber.item():.4e}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ebno_cmp = np.arange(8.0, 16.1, 1.0)

# MLP (P=2) —— 用之前存的权重
m_mlp = E2ESystemPilotCSI(P=2, L=25, training=False, perfect_csi=False).to(device)
load_weights(m_mlp, weights_by_P[2])          # P 扫描时存的 P=2 权重
with torch.no_grad():
    ber_mlp, _ = sim_ber(m_mlp, ebno_cmp, batch_size=128,
                         num_target_block_errors=300, max_mc_iter=100)

# CNN (P=2)
m_cnn = E2ESystemPilotCNN(P=2, L=25, training=False, perfect_csi=False).to(device)
load_weights(m_cnn, 'rayleigh_cnn_P2_weights')
with torch.no_grad():
    ber_cnn, _ = sim_ber(m_cnn, ebno_cmp, batch_size=128,
                         num_target_block_errors=300, max_mc_iter=100)

ber_mlp = ber_mlp.cpu().numpy()
ber_cnn = ber_cnn.cpu().numpy()

plt.figure(figsize=(9, 6))
mask = ber_mlp > 0
plt.semilogy(ebno_cmp[mask], ber_mlp[mask], 'o-', label='MLP demapper (P=2)')
mask = ber_cnn > 0
plt.semilogy(ebno_cmp[mask], ber_cnn[mask], 's--', label='CNN demapper (P=2)')
plt.axhline(1e-2, color='gray', ls=':', label='target 1e-2')
plt.xlabel(r'$E_b/N_0$ (dB)')
plt.ylabel('BER')
plt.grid(which='both', alpha=0.4)
plt.legend()
plt.title('MLP vs CNN demapper, block-fading + pilot CSI (P=2, L=25)')
plt.tight_layout()
plt.savefig('ber_mlp_vs_cnn.png', dpi=150)
plt.show()